# Protein sequence design from a masked sequence with ESM-2 (650M)

This notebook designs (in-paints) the masked positions of a protein sequence using
[ESM-2](https://github.com/facebookresearch/esm) **650M** (`esm2_t33_650M_UR50D`, ~650 MB)
as a masked language model.

It implements **three** design strategies, separately:

1. **Naive single-pass fill** — one forward pass; decode all masked positions independently (matches `archive/design.py`).
2. **Iterative / autoregressive decoding** — fill one position at a time, re-running the model after each commit so later predictions are conditioned on earlier ones. Usually more self-consistent.
3. **Scoring & ranking by pseudo-perplexity** — generate many candidates and rank them by ESM-2 masked-marginal pseudo-log-likelihood (pseudo-perplexity).

> **ESMFold vs. ESM-2.** ESMFold predicts 3D *structure* from sequence — it does **not** design sequence. Filling masked residues is a masked-LM task, which is what the ESM-2 LM head does here.

## 0. Setup

Install dependencies (uncomment if needed). The 650 MB weights download to the torch hub cache on first model load.

In [ ]:
# !pip install fair-esm torch

In [ ]:
import math
from dataclasses import dataclass

import torch
import esm  # fair-esm

MODEL_NAME = "esm2_t33_650M_UR50D"
STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"
MASK_CHAR = "#"  # marks positions to design in a plain string


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("device:", DEVICE)

## 1. Load the ESM-2 650M model

First run downloads ~650 MB. `model`, `alphabet`, and `batch_converter` are reused by all three strategies.

In [ ]:
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model = model.eval().to(DEVICE)
batch_converter = alphabet.get_batch_converter()
print(f"loaded {MODEL_NAME}: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params")

## 2. Shared helpers

Tokenization, a boolean mask restricting predictions to the 20 standard amino acids, and reconstruction of a residue-only sequence from token ids.

In [ ]:
def standard_aa_mask(alphabet):
    """Boolean tensor over the vocab selecting the 20 standard amino acids."""
    keep = torch.zeros(len(alphabet.all_toks), dtype=torch.bool)
    for aa in STANDARD_AA:
        keep[alphabet.get_idx(aa)] = True
    return keep


AA_MASK = standard_aa_mask(alphabet).to(DEVICE)
SPECIAL_IDX = {alphabet.cls_idx, alphabet.eos_idx, alphabet.padding_idx, alphabet.mask_idx}


def to_tokens(seq_with_mask: str):
    """Plain string with MASK_CHAR -> ESM token tensor on DEVICE.

    '#' is mapped to ESM's '<mask>' token. A leading BOS/cls and trailing EOS
    are added by the batch converter.
    """
    esm_seq = seq_with_mask.strip().upper().replace(MASK_CHAR, "<mask>")
    _, _, tokens = batch_converter([("query", esm_seq)])
    return tokens.to(DEVICE)


def tokens_to_seq(tokens_1d: torch.Tensor) -> str:
    """Token ids (1D, incl. specials) -> residue-only string."""
    return "".join(
        alphabet.get_tok(int(t)) for t in tokens_1d if int(t) not in SPECIAL_IDX
    )


# A demo target: a short sequence with 4 contiguous masked positions.
DEMO = "MKTAYIAKQR" + MASK_CHAR * 4 + "GFTLLILVDDDEK"
print("demo (masked):", DEMO)

---
## Strategy 1 — Naive single-pass fill

Run **one** forward pass and decode every masked position from that single distribution. Each masked
position is decoded independently (greedy `argmax`, or temperature/top-k sampling). Fast, but masked
positions don't "see" each other's chosen residues — fine for sparse/independent masks, weaker for long
contiguous spans.

In [ ]:
@dataclass
class DesignResult:
    sequence: str
    filled: list      # list of (pos_0based_in_residues, aa, prob)
    mean_logp: float  # mean log-prob of the filled residues


@torch.no_grad()
def design_naive(seq_with_mask, temperature=0.0, top_k=None,
                 restrict_standard=True, seed=None):
    """Single forward pass; decode all masked positions independently."""
    if seed is not None:
        torch.manual_seed(seed)

    tokens = to_tokens(seq_with_mask)
    logits = model(tokens)["logits"][0]  # (L, vocab)
    if restrict_standard:
        logits = logits.masked_fill(~AA_MASK, float("-inf"))

    mask_positions = (tokens[0] == alphabet.mask_idx).nonzero(as_tuple=True)[0]
    full = tokens[0].clone()
    filled, logp_sum = [], 0.0

    for pos in mask_positions:
        pos_logits = logits[pos]
        probs = torch.softmax(pos_logits, dim=-1)
        if temperature and temperature > 0:
            scaled = pos_logits / temperature
            if top_k:
                topv, topi = torch.topk(scaled, k=min(top_k, scaled.numel()))
                filt = torch.full_like(scaled, float("-inf"))
                filt[topi] = topv
                scaled = filt
            choice = torch.multinomial(torch.softmax(scaled, dim=-1), 1).item()
        else:
            choice = int(torch.argmax(pos_logits).item())
        full[pos] = choice
        filled.append((int(pos.item()) - 1, alphabet.get_tok(choice), float(probs[choice])))
        logp_sum += math.log(float(probs[choice]) + 1e-9)

    return DesignResult(tokens_to_seq(full), filled, logp_sum / max(len(filled), 1))

In [ ]:
# Greedy naive fill
res = design_naive(DEMO, temperature=0.0)
print("designed:", res.sequence)
print("mean_logp:", round(res.mean_logp, 3))
print("filled   :", "  ".join(f"{p+1}{aa}({pr:.2f})" for p, aa, pr in res.filled))

In [ ]:
# A few sampled naive designs
for i in range(3):
    r = design_naive(DEMO, temperature=1.0, seed=i)
    print(f"sample {i}: {r.sequence}   mean_logp={r.mean_logp:.3f}")

---
## Strategy 2 — Iterative / autoregressive decoding

Fill masked positions **one at a time**, re-running the model after each commit so every new prediction
is conditioned on the residues already chosen. Two ordering policies:

- `order="confidence"` (default): at each step, fill the **most confident** remaining masked position (max-probability decoding, à la MaskGIT). Robust, order-free.
- `order="left_to_right"`: classic autoregressive left→right fill.

Costs *k* forward passes for *k* masked positions, but typically yields more self-consistent spans than the naive single pass.

In [ ]:
@torch.no_grad()
def design_iterative(seq_with_mask, order="confidence", temperature=0.0,
                     top_k=None, restrict_standard=True, seed=None):
    """Decode masked positions one at a time, re-running the model each step."""
    if seed is not None:
        torch.manual_seed(seed)
    assert order in ("confidence", "left_to_right")

    tokens = to_tokens(seq_with_mask)
    full = tokens[0].clone()
    remaining = (full == alphabet.mask_idx).nonzero(as_tuple=True)[0].tolist()
    filled, logp_sum, n = [], 0.0, len(remaining)

    while remaining:
        logits = model(full.unsqueeze(0))["logits"][0]
        if restrict_standard:
            logits = logits.masked_fill(~AA_MASK, float("-inf"))
        probs = torch.softmax(logits, dim=-1)

        if order == "left_to_right":
            pos = remaining[0]
        else:  # confidence: pick the remaining mask whose top prob is highest
            top_per_pos = {p: float(probs[p].max()) for p in remaining}
            pos = max(top_per_pos, key=top_per_pos.get)

        pos_logits = logits[pos]
        if temperature and temperature > 0:
            scaled = pos_logits / temperature
            if top_k:
                topv, topi = torch.topk(scaled, k=min(top_k, scaled.numel()))
                filt = torch.full_like(scaled, float("-inf"))
                filt[topi] = topv
                scaled = filt
            choice = torch.multinomial(torch.softmax(scaled, dim=-1), 1).item()
        else:
            choice = int(torch.argmax(pos_logits).item())

        full[pos] = choice
        filled.append((pos - 1, alphabet.get_tok(choice), float(probs[pos, choice])))
        logp_sum += math.log(float(probs[pos, choice]) + 1e-9)
        remaining.remove(pos)

    filled.sort(key=lambda x: x[0])  # report in sequence order
    return DesignResult(tokens_to_seq(full), filled, logp_sum / max(n, 1))

In [ ]:
for ordering in ("confidence", "left_to_right"):
    r = design_iterative(DEMO, order=ordering, temperature=0.0)
    print(f"[{ordering:13s}] {r.sequence}   mean_logp={r.mean_logp:.3f}")
    print("               filled:", "  ".join(f"{p+1}{aa}({pr:.2f})" for p, aa, pr in r.filled))

---
## Strategy 3 — Scoring & ranking by pseudo-perplexity

An MLM has no single likelihood for a sequence, so we use the **masked-marginal pseudo-log-likelihood**
(PLL): mask each residue in turn, read off the model's log-prob for the true residue, and sum.

$$\text{PLL}(x)=\sum_{i} \log p_\theta(x_i \mid x_{\setminus i}), \qquad \text{pseudo-perplexity}=\exp\!\left(-\tfrac{1}{L}\,\text{PLL}(x)\right)$$

Lower pseudo-perplexity = the model finds the sequence more "natural." We generate many candidate designs
(naive and/or iterative) and rank them. To keep it efficient we score in batches: one masked copy per position.

In [ ]:
@torch.no_grad()
def pseudo_perplexity(sequence, batch_size=32):
    """Masked-marginal pseudo-perplexity of a (fully resolved) residue string.

    Masks each residue position once, batches the masked copies, and reads the
    log-prob of the original residue. Returns (pll, pseudo_perplexity).
    """
    _, _, base = batch_converter([("seq", sequence.strip().upper())])
    base = base[0].to(DEVICE)  # (L,) incl. BOS/EOS
    res_positions = [i for i, t in enumerate(base.tolist()) if t not in SPECIAL_IDX]

    total_logp = 0.0
    for start in range(0, len(res_positions), batch_size):
        chunk = res_positions[start:start + batch_size]
        batch = base.unsqueeze(0).repeat(len(chunk), 1).clone()
        for row, pos in enumerate(chunk):
            batch[row, pos] = alphabet.mask_idx
        logits = model(batch)["logits"]
        logprobs = torch.log_softmax(logits, dim=-1)
        for row, pos in enumerate(chunk):
            total_logp += float(logprobs[row, pos, base[pos]])

    n = len(res_positions)
    return total_logp, math.exp(-total_logp / n)

In [ ]:
# Sanity check: score the greedy naive and greedy iterative designs.
for name, seq in [
    ("naive-greedy", design_naive(DEMO, temperature=0.0).sequence),
    ("iter-confidence", design_iterative(DEMO, order="confidence").sequence),
]:
    pll, ppl = pseudo_perplexity(seq)
    print(f"{name:16s} ppl={ppl:6.3f}  pll={pll:8.2f}  {seq}")

### Generate-and-rank pipeline

Combine the strategies: draw a pool of candidates (sampled naive + sampled iterative + the two greedy
anchors), deduplicate, and rank by pseudo-perplexity.

In [ ]:
def generate_and_rank(seq_with_mask, n_naive=8, n_iter=8, temperature=1.0,
                      top_k=5, score_batch=32):
    """Build a candidate pool from both strategies and rank by pseudo-perplexity."""
    candidates = {}  # sequence -> source label
    candidates[design_naive(seq_with_mask, temperature=0.0).sequence] = "naive-greedy"
    candidates.setdefault(
        design_iterative(seq_with_mask, order="confidence").sequence, "iter-greedy"
    )
    for i in range(n_naive):
        s = design_naive(seq_with_mask, temperature=temperature, top_k=top_k, seed=1000 + i).sequence
        candidates.setdefault(s, "naive-sample")
    for i in range(n_iter):
        s = design_iterative(seq_with_mask, order="confidence", temperature=temperature,
                             top_k=top_k, seed=2000 + i).sequence
        candidates.setdefault(s, "iter-sample")

    ranked = []
    for seq, src in candidates.items():
        pll, ppl = pseudo_perplexity(seq, batch_size=score_batch)
        ranked.append((ppl, pll, src, seq))
    ranked.sort(key=lambda x: x[0])  # ascending pseudo-perplexity = best first
    return ranked


ranked = generate_and_rank(DEMO, n_naive=8, n_iter=8, temperature=1.0, top_k=5)
print(f"{'rank':>4}  {'ppl':>7}  {'source':14s}  sequence")
for i, (ppl, pll, src, seq) in enumerate(ranked, 1):
    print(f"{i:>4}  {ppl:7.3f}  {src:14s}  {seq}")

### Best design

In [ ]:
best_ppl, best_pll, best_src, best_seq = ranked[0]
print("best source        :", best_src)
print("pseudo-perplexity  :", round(best_ppl, 3))
print("sequence           :", best_seq)

---
## Notes & extensions

- **Use your own target:** set `DEMO = "..."` with `#` at every position to design, then re-run the strategy cells. Sparse single masks favor the naive pass; long contiguous spans favor iterative decoding.
- **Diversity vs. quality:** raise `temperature` / `top_k` for more diverse candidates, then let pseudo-perplexity ranking pick the best.
- **Speed:** pseudo-perplexity costs ~L forward passes per sequence (batched here). For large pools, reduce the pool, raise `score_batch`, or score on GPU.
- **Non-standard residues:** pass `restrict_standard=False` to allow tokens like B/U/Z/O/X.
- **Structure:** to evaluate a designed sequence's fold, run it through ESMFold separately — that is a distinct structure-prediction step, not part of sequence design.